In [ ]:

import hashlib, json, shutil, urllib.request, zipfile
from collections import Counter
from pathlib import Path
url = 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N3DH-CE.zip'
archive = Path('/tmp/Fluo-N3DH-CE.zip')
digest = hashlib.sha256()
with urllib.request.urlopen(url, timeout=90) as response, archive.open('wb') as target:
    while chunk := response.read(1024 * 1024):
        digest.update(chunk)
        target.write(chunk)
with zipfile.ZipFile(archive) as data:
    paths = [Path(n) for n in data.namelist() if not n.endswith('/')]
    files = Counter(str(n.parent) for n in paths)
    lineage_files = [str(n).replace(chr(92), '/') for n in paths if n.name == 'man_track.txt']
    lineage_rows = {n: len(data.read(n).decode().splitlines()) for n in lineage_files}
report = {'source': url, 'sha256': digest.hexdigest(), 'bytes': archive.stat().st_size,
          'files_by_directory': dict(files), 'lineage_rows': lineage_rows, 'optimizer_steps': 0}
Path('/kaggle/working/data_audit.json').write_text(json.dumps(report, indent=2))
print(report)
